In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Pendahuluan 
Input semua data, yaitu train, test, dan sample. Kemudian, Buat flasifikasi setiap datanya agar dapat terbaca dengan jelas dan rapih berdasarkan urutan data yang memiliki pengaruh lebih banyak dengan membandingkan ketiga data.

In [ ]:
#import libraries support
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler


file_path = "/kaggle/input/house-prices-advanced-regression-techniques/train.csv"
test_path = "/kaggle/input/house-prices-advanced-regression-techniques/test.csv"
sample_path = "/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv"

home_data = pd.read_csv(file_path)
test_data = pd.read_csv(test_path)
sample_data = pd.read_csv(sample_path)

home_data.shape, test_data.shape, sample_data.head()
train_y = home_data['SalePrice']
test_id = test_data['Id']
train_id = home_data['Id']

# FEATURE ENGINEERING
Diperlukan feature baru untuk memperoleh hasil yang lebih akurat. Feature baru tersebut berupa:

1. TotalBath = BsmtFullBath + (0.5 * BsmtHalfBath) + FullBath + (0.5 * HalfBath)
2. TotalSF = TotalBmstSF + 1stFlrSF + 2ndFlrSF
3. TotalPorch = WoodDeckSF  + OpenPorchSF  + EnclosedPorch  + 3SsnPorch  + ScreenPorch
4. OverallQual_Sq = OverallQual ** 2
5. GrLivArea_Sq = GrLivArea ** 2
6. abvGradeQual = OverallQual * GrLivArea
7. spaceQuality = Total_SqFt * OverallQual
8. TotalQuality = OverallQual + OverallCond
9. Age = YrSold - YrBuilt
10. Age_Remod = YrSold - YearRemodAdd
11. Garage_Age = YrSold - GarageYrBlt

Boolean Check - (1-iya, 0-tidak):
1. Remodeled (YearRemodAdd != YearBuilt) 
2. New (YearBuilt == YrSold)
3. HasPool (PoolArea > 0)
4. HasBsmt (TotalBsmtSF > 0)
5. HasFireplace (Fireplaces > 0)
6. HasGarage (GarageArea > 0)

Cara mengimplementasikannya adalah menggabungkan terlebih dahulu (concatenate) train.csv dan test.csv, kemudian membuat feature baru di kedua data, dan memisahkan kedua data tersebut kembali.

In [ ]:
home_data = home_data.drop(['Id', 'SalePrice'], axis=1)
test_data = test_data.drop('Id', axis=1)

trainLength = len(home_data)

allData = pd.concat([home_data, test_data], axis=0)


# ubah null jadi 0 utk numerical data, kecuali lotfrontage sama garageyrblt
numData = allData.select_dtypes(include=np.number).columns
exceptionNum = ['LotFrontage', 'GarageYrBlt'] #ra boleh 0
for i in numData:
    if i not in exceptionNum:
        allData[i] = allData[i].fillna(0)
# ubah null jadi 'None' utk categorical data,
catData = allData.select_dtypes(include="object")
exceptionCat = ['MSZoning', 'Utilities', 'Exterior1st', 'Exterior2nd', 'Electrical', 'KitchenQual', 'Functional', 'SaleType']
for i in catData:
    if i not in exceptionCat:
        allData[i] = allData[i].fillna("None")

#isi exceptionNum
allData['LotFrontage'] = allData['LotFrontage'].fillna(allData['LotFrontage'].median())
allData['GarageYrBlt'] = allData['GarageYrBlt'].fillna(allData['YrSold'])
#isi exceptionCat
for i in exceptionCat:
    if i in allData.columns:
        if allData[i].isnull().any():
            allData[i] = allData[i].fillna(allData[i].mode()[0]) 

# end

#yaaa mulai feature engineeringnya
allData['TotalBath'] = allData['BsmtFullBath'] + (0.5 * allData['BsmtHalfBath']) + allData['FullBath'] + (0.5 * allData['HalfBath'])
allData['TotalSF'] = allData['TotalBsmtSF'] + allData['1stFlrSF'] + allData['2ndFlrSF']
allData['TotalPorch'] = allData['WoodDeckSF'] + allData['OpenPorchSF'] + allData['EnclosedPorch'] + allData['3SsnPorch'] + allData['ScreenPorch']
allData['OverallQual_Sq'] = allData['OverallQual'] ** 2
allData['GrLivArea_Sq'] = allData['GrLivArea'] ** 2
allData['abvGradeQual'] = allData['OverallQual'] * allData['GrLivArea']
allData['spaceQuality'] = allData['TotalSF'] * allData['OverallQual']
allData['TotalQuality'] = allData['OverallQual'] + allData['OverallCond']
allData['Age'] = allData['YrSold'] - allData['YearBuilt']
allData['Age_Remod'] = allData['YrSold'] - allData['YearRemodAdd']
allData['Garage_Age'] = allData['YrSold'] - allData['GarageYrBlt']
#boolean
allData['Remodeled'] = (allData['YearRemodAdd'] != allData['YearBuilt']).astype(int)
allData['New'] = (allData['YearBuilt'] == allData['YrSold']).astype(int)
allData['HasPool'] = (allData['PoolArea'] > 0).astype(int)
allData['HasBsmt'] = (allData['TotalBsmtSF'] > 0).astype(int)
allData['HasFireplace'] = (allData['Fireplaces'] > 0).astype(int)
allData['HasGarage'] = (allData['GarageArea'] > 0).astype(int)
#drop data
dropData = ['BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'YrSold', 'YearBuilt',
            'YearRemodAdd', 'PoolArea', 'Fireplaces', 'GarageArea', 'GarageYrBlt']
allData = allData.drop(columns=dropData)

trainTemp = allData.iloc[:trainLength]
testTemp = allData.iloc[trainLength:]


Check kembali data dari train

In [ ]:
trainTemp.describe()

# MENENTUKAN FEATURES YANG TERBAIK
Akan kita bagi menjadi dua, yaitu numerical value dan categorical value

# NUMERICAL

In [ ]:
#periksa kolom apa saja yang menggunakan numerical value
trainTemp.describe().columns 

In [ ]:
trainTempNum = trainTemp[['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond',
       'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'LowQualFinSF',
       'GrLivArea', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd',
       'GarageCars', 'MiscVal', 'MoSold', 'TotalBath', 'TotalSF', 'TotalPorch',
       'OverallQual_Sq', 'GrLivArea_Sq', 'abvGradeQual',
       'spaceQuality', 'TotalQuality', 'Age', 'Age_Remod',
       'Garage_Age', 'Remodeled', 'New', 'HasPool', 'HasBsmt', 'HasFireplace',
       'HasGarage']]

heatmapData = trainTempNum.copy()
heatmapData['SalePrice'] = train_y 
numerical_heatmap_data = heatmapData.select_dtypes(include=np.number)
corr_matrix = numerical_heatmap_data.corr()
sale_corr = corr_matrix[['SalePrice']] 
sale_corr_sorted = sale_corr.sort_values(by='SalePrice', ascending=False)
plt.figure(figsize=(10, 20)) 
sns.heatmap(
    sale_corr_sorted, 
    annot=True,     
    cmap='coolwarm',
    fmt='.2f'  
)
plt.show()

abs_corr = sale_corr_sorted['SalePrice'].abs()
strong_corr_series = abs_corr[abs_corr > 0.20]
final_numerical_features = strong_corr_series.index.tolist()

if 'SalePrice' in final_numerical_features:
    final_numerical_features.remove('SalePrice')

print("Fitur Numerik Final (Korelasi > 0.20):")
print(final_numerical_features)

Features yang harus kita gunakan adalah yang lebih besar dari pada 0.2 yaitu:

['spaceQuality', 'abvGradeQual', 'OverallQual_Sq', 'OverallQual', 'TotalSF', 'GrLivArea', 'GrLivArea_Sq', 'GarageCars', 'TotalBath', 'TotalQuality', 'TotRmsAbvGrd', 'MasVnrArea', 'HasFireplace', 'TotalPorch', 'BsmtFinSF1', 'LotFrontage', 'LotArea', 'HasGarage', 'New', 'BsmtUnfSF', 'Garage_Age', 'Age_Remod', 'Age']

kita juga dapat menyimpan kolom dengan value di bawah -0.2, karena kolom tersebut menurunkan harga sehingga berguna utk akurasi harga

['Garage_Age', 'Age_Remod', 'Age']

In [ ]:
numFeatures = ['Garage_Age', 'Age_Remod', 'Age','spaceQuality', 'abvGradeQual', 'OverallQual_Sq', 'OverallQual', 'TotalSF', 'GrLivArea', 'GrLivArea_Sq', 'GarageCars', 'TotalBath', 'TotalQuality', 'TotRmsAbvGrd', 'MasVnrArea', 'HasFireplace', 'TotalPorch', 'BsmtFinSF1', 'LotFrontage', 'LotArea', 'HasGarage', 'New', 'BsmtUnfSF', 'Garage_Age', 'Age_Remod', 'Age']

# CATEGORICAL

In [ ]:
print(trainTemp.select_dtypes(include=['object']).columns)

In [ ]:
trainTempCat = [['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition']]

cari feature yang harus kita keep menggunakan mutual information 

In [ ]:
categorical_column_names = trainTemp.select_dtypes(include='object').columns.tolist()

copyCat_DataFrame = trainTemp[categorical_column_names].copy()

copyCat_DataFrame = copyCat_DataFrame.fillna("None")

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
copyCat_encoded = encoder.fit_transform(copyCat_DataFrame)

mi_scores = mutual_info_regression(copyCat_encoded, heatmapData['SalePrice']) 

mi_series = pd.Series(mi_scores, index=categorical_column_names) 

mi_series_sorted = mi_series.sort_values(ascending=False)
print(mi_series_sorted)

keep features yang memiliki angka lebih dari 0.051 yaitu:
['Neighborhood', 'ExterQual', 'BsmtQual', 'KitchenQual', 'GarageFinish', 'GarageType', 'FireplaceQu', 'Foundation', 'Exterior2nd', 'HeatingQC', 'BsmtFinType1', 'MSZoning', 'Exterior1st', 'MasVnrType', 'LotShape', 'BsmtExposure', 'SaleType', 'SaleCondition', 'HouseStyle', 'GarageQual', 'CentralAir', 'GarageCond', 'Electrical', 'BldgType' ]

In [ ]:
catFeatures = ['Neighborhood', 'ExterQual', 'BsmtQual', 'KitchenQual', 'GarageFinish', 'GarageType', 'FireplaceQu', 'Foundation', 'Exterior2nd', 'HeatingQC', 'BsmtFinType1', 'MSZoning', 'Exterior1st', 'MasVnrType', 'LotShape', 'BsmtExposure', 'SaleType', 'SaleCondition', 'HouseStyle', 'GarageQual', 'CentralAir', 'GarageCond', 'Electrical', 'BldgType' ]

In [ ]:
scaler = StandardScaler()
num_data = allData[numFeatures]
num_data_scaled = scaler.fit_transform(num_data)
num_processed_df = pd.DataFrame(num_data_scaled, 
                                columns=numFeatures, 
                                index=allData.index)

cat_data = allData[catFeatures]
cat_processed_df = pd.get_dummies(cat_data, drop_first=True)

final_data_processed = pd.concat([num_processed_df, cat_processed_df], axis=1)

# 

# Input Nilai Features
Input data features yang digunakan dengan pertimbangan 


In [ ]:
X_train_final = final_data_processed.iloc[:trainLength]
X_test_final = final_data_processed.iloc[trainLength:]
y_train_log = np.log1p(train_y)


# Train Nilai x dan y
Train untuk setiap nilai x dan y 

In [ ]:
train_x, val_x, train_y, val_y = train_test_split(X_train_final, y_train_log, random_state = 1)

# Latih Model Menggunakan RandomForestRegressor untuk Cek Nilai MAE Awal

In [ ]:
rf_model = RandomForestRegressor(random_state = 1)
rf_model.fit(train_x, train_y)
rf_val_pred = rf_model.predict(val_x)
rf_val_mae = mean_absolute_error(rf_val_pred, val_y)

print("Validation MAE for Random Forest Model: {:,.0f}".format(rf_val_mae))

# Mencari Nilai MAE Terendah
Mencari nilai MAE terendah dengan menggunakan n-estimator untuk pengecekan setiap faktor

In [ ]:
"""
def get_mae(max_n, train_x, val_x, train_y, val_y):
    model = RandomForestRegressor(n_estimators=max_n, random_state=1)
    model.fit(train_x, train_y)
    preds_val = model.predict(val_x)
    mae = mean_absolute_error(val_y, preds_val)
    return mae

n = [10, 100, 1000, 10000, 50000]
for max_n in n:
    n_mae = get_mae(max_n, train_x, val_x, train_y, val_y)
    print("Max n estimator: %d  \t\t Mean Absolute Error:  %d" %(max_n, n_mae))
"""

# Final Data dan Tes Data
Tes data menggunakan n estimator yang sudah didapat dengan nilai mae terkecil

In [ ]:
final_model = RandomForestRegressor(random_state=1, n_estimators=1)
final_model.fit(X_train_final, y_train_log)
log_predictions = final_model.predict(X_test_final)
final_predictions = np.expm1(log_predictions)


# Submission

In [ ]:
test_data_original = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv')

submission = pd.DataFrame({
    "Id": test_data_original.Id,  # Gunakan 'Id' dari file asli
    "SalePrice": final_predictions
})

submission.to_csv('submission.csv', index=False)
print("success")